# broadcasting-rules — ex5: outer product via column×row broadcast

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcasting-rules`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five broadcasting patterns that ramp from predicting the result shape → row-vector broadcast → column-vector broadcast → targeted axis insertion → outer product via broadcast. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `broadcasting-rules`**, which bridges to the bank subtopic `Numpy: Vectorization and broadcasting` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Broadcasting — quick refresher

**The rule** (NumPy & PyTorch agree):
1. Right-align both shapes; left-pad the shorter with 1s.
2. For each pair of aligned axes: equal → keep; one is 1 → use the other; otherwise → incompatible.

**Three patterns you reach for constantly:**
- **Row broadcast** — `(N, D) + (D,)` works automatically. Adds a per-feature bias.
- **Column broadcast** — `(N, D) * w` where `w` is `(N,)` fails. Reshape `w` to `(N, 1)` first.
- **Axis insertion** — `unsqueeze` / `[:, None]` / `reshape` are all valid ways to insert a size-1 axis where broadcasting needs it.

### Exercise 5 — outer product via column×row broadcast

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize column-vector broadcast + row-vector broadcast to compute the outer product of two 1-D tensors without `torch.outer`.
> Keywords: outer-product, column-times-row, integration, multi-kc
> ```

**KCs targeted:** `broadcast-row-vector`, `broadcast-column-vector`, `broadcast-outer-product`

Implement `ex5_outer(u, v)` to compute the outer product of two 1-D tensors.

Input shapes: `u` is `(N,)`, `v` is `(M,)`. Output shape: `(N, M)`. Each `out[i, j] == u[i] * v[j]`.

**Use broadcasting only** — no `torch.outer`, no `einsum`, no `unsqueeze` + matmul. Strategy: reshape `u` to a column `(N, 1)` and `v` to a row `(1, M)`, then multiply. The right-align rule produces `(N, M)`.

Equivalent to `torch.outer(u, v)`.

> ⚠️ **Integrative exercise.** This combines 3+ KCs (column-broadcast, row-broadcast, axis insertion) in one expression; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_outer(u: Tensor, v: Tensor) -> Tensor:
    return u.unsqueeze(1) * v.unsqueeze(0)


<details><summary>Solution</summary>

```python
def ex5_outer(u: Tensor, v: Tensor) -> Tensor:
    return u.unsqueeze(1) * v.unsqueeze(0)
```

**Reading the pattern.**
- `u.unsqueeze(1)` → shape `(N, 1)` (column vector).
- `v.unsqueeze(0)` → shape `(1, M)` (row vector).
- `(N, 1) * (1, M)` right-aligns as `(N, 1) * (1, M)` → broadcasts to `(N, M)`.

Each broadcast tile fills a row (from `v`) or a column (from `u`); their elementwise product gives `u[i] * v[j]` at every position.

**Equivalent forms:** `u[:, None] * v[None, :]` (same thing, slice syntax); `torch.einsum('i,j->ij', u, v)`; `torch.outer(u, v)`. The broadcasting form is worth knowing because it generalizes to higher ranks (e.g. batched outer product) without changing the mental model.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex5',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()